<a href="https://colab.research.google.com/github/sameerahmed-cloud/text2sql-small-llm/blob/main/notebooks/01_get_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
REPO = "/content/text2sql-small-llm"
if os.path.exists(REPO):
    !git -C {REPO} pull          # folder exists: just update it
else:
    !git clone https://github.com/sameerahmed-cloud/text2sql-small-llm.git {REPO}
%cd {REPO}
!git log --oneline -1

Mounted at /content/drive
Cloning into '/content/text2sql-small-llm'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 48 (delta 15), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 18.45 KiB | 2.05 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/text2sql-small-llm
fd990c7 (HEAD -> main, origin/main, origin/HEAD) Add containment check to leak detector


In [2]:
# Cell 2: download once (safe to re-run; finished files are skipped)
from src.get_data import download_all
download_all()

✓ bird_dev: already stored (Drive), skipping
✓ spider: already stored (Drive), skipping
✓ bird_train: already stored (hf:Sameer-ahmed/text2sql-raw), skipping

Manifest:
{
  "bird_dev": {
    "file": "bird_dev.zip",
    "source": "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip",
    "bytes": 346207293,
    "sha256": "cdd6d19faeb45a23970b98d3ef6c40a87987c95459c2cf12076897a60cf5a630",
    "license": "CC BY-SA 4.0",
    "downloaded": "2026-11-09"
  },
  "spider": {
    "file": "spider_data.zip",
    "source": "gdrive:1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J",
    "bytes": 205800266,
    "sha256": "00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b",
    "license": "CC BY-SA 4.0",
    "downloaded": "2026-11-09"
  },
  "bird_train": {
    "file": "bird_train.zip",
    "source": "https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip",
    "stored_in": "hf:Sameer-ahmed/text2sql-raw",
    "bytes": 8919543554,
    "sha256": "66e9e3115b59559554013aa3b124156249f30437a6b4e4f96de3d

{'bird_dev': {'file': 'bird_dev.zip',
  'source': 'https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip',
  'bytes': 346207293,
  'sha256': 'cdd6d19faeb45a23970b98d3ef6c40a87987c95459c2cf12076897a60cf5a630',
  'license': 'CC BY-SA 4.0',
  'downloaded': '2026-11-09'},
 'spider': {'file': 'spider_data.zip',
  'source': 'gdrive:1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J',
  'bytes': 205800266,
  'sha256': '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b',
  'license': 'CC BY-SA 4.0',
  'downloaded': '2026-11-09'},
 'bird_train': {'file': 'bird_train.zip',
  'source': 'https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip',
  'stored_in': 'hf:Sameer-ahmed/text2sql-raw',
  'bytes': 8919543554,
  'sha256': '66e9e3115b59559554013aa3b124156249f30437a6b4e4f96de3d2dfb5ae8cbc',
  'license': 'CC BY-SA 4.0',
  'downloaded': '2026-11-09'}}

In [3]:
# Cell 3: unpack to local disk and check the counts
from src.get_data import unpack_all, check_data
paths = unpack_all(include_bird_train=True)
check_data(paths)

Unpacking bird_dev (big datasets take several minutes)...
Unpacking spider (big datasets take several minutes)...


bird_train.zip: reconstructing file:   0%|          |  0.00B / 8.92GB            

bird_train.zip: downloading bytes:           |  0.00B            

Unpacking bird_train (big datasets take several minutes)...
✓ Paths saved to /content/data/raw/paths.json

bird_dev: 11 database folders in /content/data/raw/bird_dev/dev_20240627/dev_databases
  dev: 1534 questions over 11 databases  | missing DB folders: 0
  sample DB 'california_schools': 3 tables (opened read-only ✓)

spider: 166 database folders in /content/data/raw/spider/spider_data/database
  train: 7000 questions over 140 databases  | missing DB folders: 0
  dev: 1034 questions over 20 databases  | missing DB folders: 0
  sample DB 'academic': 15 tables (opened read-only ✓)

bird_train: 69 database folders in /content/data/raw/bird_train/train/train_databases
  train: 9428 questions over 69 databases  | missing DB folders: 0
  sample DB 'address': 9 tables (opened read-only ✓)


In [4]:
# Step 1.2: split by database
from src.data_prep import make_splits
summary = make_splits(paths)

Comparing 209 training DBs with 31 test DBs...
  review: spider/allergy_1 vs spider/pets_1 similarity 0.44, containment 0.67 (kept)
  review: spider/club_1 vs spider/pets_1 similarity 0.38, containment 0.57 (kept)
  review: spider/dorm_1 vs spider/pets_1 similarity 0.32, containment 0.57 (kept)
  review: spider/game_1 vs spider/pets_1 similarity 0.32, containment 0.57 (kept)
  review: spider/voter_2 vs spider/pets_1 similarity 0.35, containment 0.57 (kept)
  removed from training: bird_train/world — matches test DB spider/world_1 (name match: False, similarity 1.00, containment 1.00)
  removed from training: spider/formula_1 — matches test DB bird_dev/formula_1 (name match: True, similarity 1.00, containment 1.00)
  removed from training: spider/soccer_1 — matches test DB bird_dev/european_football_2 (name match: False, similarity 0.42, containment 1.00)
✓ No database appears in more than one split
✓ Every question's database file exists

Saved to /content/drive/MyDrive/text2sql/data/p

In [5]:
# Cell 4: make sure Drive has finished saving before you close the tab
drive.flush_and_unmount()